In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_metadata_snapshot as ncbi_metadata_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pca_kmeans as pca_kmeans_module
import src.pago_pipeline.pca_kmeans_snapshot as pca_kmeans_snapshot_module
import src.pago_pipeline.sweep_genes_snapshot as sweep_genes_snapshot_module
from src.pago_pipeline.storage import sha256_of_file

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_metadata_snapshot_module = importlib.reload(ncbi_metadata_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pca_kmeans_module = importlib.reload(pca_kmeans_module)
pca_kmeans_snapshot_module = importlib.reload(pca_kmeans_snapshot_module)
sweep_genes_snapshot_module = importlib.reload(sweep_genes_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
load_latest_metadata_snapshot = ncbi_metadata_snapshot_module.load_latest_metadata_snapshot
load_latest_sweep_genes_snapshot = (
    sweep_genes_snapshot_module.load_latest_sweep_genes_snapshot
)
resolve_pca_kmeans_snapshot = pca_kmeans_snapshot_module.resolve_pca_kmeans_snapshot
latest_pca_kmeans_snapshot_is_available = (
    pca_kmeans_snapshot_module.latest_pca_kmeans_snapshot_is_available
)

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 — Define PCA/KMeans snapshot configuration
# =============================================================================

METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "ncbi" / "protein_metadata_csv"
)
SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY = PROJECT_ROOT / "data" / "03-features"
PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "04-analysis" / "pca_kmeans"
)

METADATA_SNAPSHOT_MODE = SnapshotMode.reuse_latest
SWEEP_GENES_SNAPSHOT_MODE = SnapshotMode.reuse_latest
PCA_KMEANS_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create

PCA_COMPONENT_COUNT_GRID = (10, 20, 50, 100, 200)
KMEANS_CLUSTER_COUNT_GRID = tuple(range(2, 11))
PCA_SVD_SOLVER = "randomized"
PCA_RANDOM_STATE = 42
KMEANS_N_INIT = "auto"
SILHOUETTE_SAMPLE_SIZE = 8000
SILHOUETTE_RANDOM_STATE = 42
KMEANS_INITIALIZATION_REPEAT_COUNT = 6
SUBSAMPLE_REPEAT_COUNT = 6
SUBSAMPLE_FRACTION = 0.80
SUBSAMPLE_RANDOM_STATE = 123
MINIMUM_ACCEPTABLE_INIT_ARI_MIN = 0.70
MINIMUM_ACCEPTABLE_SUBSAMPLE_ARI_MIN = 0.60
EXPORT_PROJECTION_COMPONENT_COUNT = 3
UPDATE_LATEST_DIRECTORY = True

print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"SWeeP snapshot root directory: {SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA/KMeans output root directory: {PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA/KMeans snapshot mode: {PCA_KMEANS_SNAPSHOT_MODE}")
print(f"PCA component grid: {PCA_COMPONENT_COUNT_GRID}")
print(f"KMeans cluster grid: {KMEANS_CLUSTER_COUNT_GRID}")

Metadata snapshot root directory: C:\Programming\Python\pAgo-project\data\02-intermediate\ncbi\protein_metadata_csv
SWeeP snapshot root directory: C:\Programming\Python\pAgo-project\data\03-features
PCA/KMeans output root directory: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans
PCA/KMeans snapshot mode: reuse_latest_or_create
PCA component grid: (10, 20, 50, 100, 200)
KMeans cluster grid: (2, 3, 4, 5, 6, 7, 8, 9, 10)


In [4]:
# =============================================================================
# CELL 4 — Resolve active source snapshots
# =============================================================================

metadata_snapshot_payload = load_latest_metadata_snapshot(
    snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
)
sweep_genes_snapshot_payload = load_latest_sweep_genes_snapshot(
    snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
)

metadata_snapshot_directory = metadata_snapshot_payload["snapshot_directory"]
metadata_manifest_file_path = metadata_snapshot_payload["manifest_file_path"]
metadata_csv_file_path = metadata_snapshot_payload["csv_file_path"]
metadata_manifest_payload = metadata_snapshot_payload["manifest"]

sweep_genes_snapshot_directory = sweep_genes_snapshot_payload["snapshot_directory"]
sweep_genes_manifest_file_path = sweep_genes_snapshot_payload["manifest_file_path"]
sweep_genes_manifest_payload = sweep_genes_snapshot_payload["manifest"]
sweep_genes_embeddings_file_path = sweep_genes_snapshot_payload["embeddings_file_path"]
sweep_genes_sequence_metadata_file_path = sweep_genes_snapshot_payload[
    "sequence_metadata_file_path"
]

print("Resolved source snapshots successfully.")
print(f"Metadata snapshot directory: {metadata_snapshot_directory}")
print(f"Metadata CSV path: {metadata_csv_file_path}")
print(f"SWeeP snapshot directory: {sweep_genes_snapshot_directory}")
print(f"SWeeP embeddings path: {sweep_genes_embeddings_file_path}")

Resolved source snapshots successfully.
Metadata snapshot directory: C:\Programming\Python\pAgo-project\data\02-intermediate\ncbi\protein_metadata_csv\latest
Metadata CSV path: C:\Programming\Python\pAgo-project\data\02-intermediate\ncbi\protein_metadata_csv\latest\protein_metadata.csv
SWeeP snapshot directory: C:\Programming\Python\pAgo-project\data\03-features\latest
SWeeP embeddings path: C:\Programming\Python\pAgo-project\data\03-features\latest\sweep_genes_embeddings_2800D.npy


In [5]:
# =============================================================================
# CELL 5 — Resolve active PCA/KMeans snapshot
# =============================================================================

pca_kmeans_snapshot_payload = resolve_pca_kmeans_snapshot(
    snapshot_mode=PCA_KMEANS_SNAPSHOT_MODE,
    snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
    source_sweep_snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    pca_component_count_grid=PCA_COMPONENT_COUNT_GRID,
    kmeans_cluster_count_grid=KMEANS_CLUSTER_COUNT_GRID,
    pca_svd_solver=PCA_SVD_SOLVER,
    pca_random_state=PCA_RANDOM_STATE,
    kmeans_n_init=KMEANS_N_INIT,
    silhouette_sample_size=SILHOUETTE_SAMPLE_SIZE,
    silhouette_random_state=SILHOUETTE_RANDOM_STATE,
    kmeans_initialization_repeat_count=KMEANS_INITIALIZATION_REPEAT_COUNT,
    subsample_repeat_count=SUBSAMPLE_REPEAT_COUNT,
    subsample_fraction=SUBSAMPLE_FRACTION,
    subsample_random_state=SUBSAMPLE_RANDOM_STATE,
    minimum_acceptable_init_ari_min=MINIMUM_ACCEPTABLE_INIT_ARI_MIN,
    minimum_acceptable_subsample_ari_min=MINIMUM_ACCEPTABLE_SUBSAMPLE_ARI_MIN,
    export_projection_component_count=EXPORT_PROJECTION_COMPONENT_COUNT,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

pca_kmeans_snapshot_directory = pca_kmeans_snapshot_payload["snapshot_directory"]
pca_kmeans_manifest_file_path = pca_kmeans_snapshot_payload["manifest_file_path"]
pca_kmeans_manifest_payload = pca_kmeans_snapshot_payload["manifest"]
pca_coordinates_file_path = pca_kmeans_snapshot_payload["pca_coordinates_file_path"]
explained_variance_ratio_file_path = pca_kmeans_snapshot_payload[
    "explained_variance_ratio_file_path"
]
cluster_assignments_file_path = pca_kmeans_snapshot_payload[
    "cluster_assignments_file_path"
]
stability_grid_file_path = pca_kmeans_snapshot_payload["stability_grid_file_path"]
profiling_log_file_path = pca_kmeans_snapshot_payload["profiling_log_file_path"]
alignment_report_file_path = pca_kmeans_snapshot_payload["alignment_report_file_path"]
pca_coordinates = pca_kmeans_snapshot_payload["pca_coordinates"]
explained_variance_ratio = pca_kmeans_snapshot_payload["explained_variance_ratio"]
cluster_assignments_dataframe = pca_kmeans_snapshot_payload["cluster_assignments"]
stability_grid_dataframe = pca_kmeans_snapshot_payload["stability_grid"]
profiling_log_dataframe = pca_kmeans_snapshot_payload["profiling_log"]
alignment_report = pca_kmeans_snapshot_payload["alignment_report"]

print("Resolved PCA/KMeans snapshot successfully.")
print(f"Snapshot directory: {pca_kmeans_snapshot_directory}")
print(f"PCA coordinates path: {pca_coordinates_file_path}")
print(f"Cluster assignments path: {cluster_assignments_file_path}")
print(f"Stability grid path: {stability_grid_file_path}")

m= 10, k= 2 | sil=0.4271 | initARI(min/mean)=-0.098/0.480 | subARI(min/mean)=-0.045/0.382
m= 10, k= 3 | sil=0.4734 | initARI(min/mean)=0.497/0.832 | subARI(min/mean)=0.264/0.753
m= 10, k= 4 | sil=0.4024 | initARI(min/mean)=0.252/0.505 | subARI(min/mean)=0.163/0.473
m= 10, k= 5 | sil=0.3641 | initARI(min/mean)=0.220/0.514 | subARI(min/mean)=0.245/0.533
m= 10, k= 6 | sil=0.2874 | initARI(min/mean)=0.221/0.508 | subARI(min/mean)=0.309/0.545
m= 10, k= 7 | sil=0.3217 | initARI(min/mean)=0.411/0.621 | subARI(min/mean)=0.230/0.556
m= 10, k= 8 | sil=0.2376 | initARI(min/mean)=0.307/0.546 | subARI(min/mean)=0.482/0.626
m= 10, k= 9 | sil=0.2347 | initARI(min/mean)=0.253/0.547 | subARI(min/mean)=0.480/0.629
m= 10, k=10 | sil=0.2965 | initARI(min/mean)=0.587/0.727 | subARI(min/mean)=0.471/0.637
m= 20, k= 2 | sil=0.3778 | initARI(min/mean)=-0.014/0.490 | subARI(min/mean)=-0.045/0.393
m= 20, k= 3 | sil=0.4126 | initARI(min/mean)=0.257/0.769 | subARI(min/mean)=-0.096/0.400
m= 20, k= 4 | sil=0.2569 | 

C:\Programming\Python\pAgo-project\src\pago_pipeline\pca_kmeans_snapshot.py:958: DtypeWarning: Columns (0: gbseq__secondary_accessions__secondary_accn, 1: taxonomy__09, 2: taxonomy__10, 3: reference__consortium, 4: feature__cds__qual__db_xref, 5: feature__cds__qual__gene, 6: feature__cds__qual__gene_synonym, 7: feature__cds__qual__old_locus_tag, 8: feature__gene__interval__accession, 9: feature__gene__location, 10: feature__gene__qual__g_o_function, 11: feature__gene__qual__gene, 12: feature__gene__qual__gene_synonym, 13: feature__gene__qual__locus_tag, 14: feature__het__interval__accession, 15: feature__het__interval__point, 16: feature__het__location, 17: feature__het__qual__heterogen, 18: feature__non_std_res__interval__accession, 19: feature__non_std_res__interval__point, 20: feature__non_std_res__location, 21: feature__non_std_res__qual__non_std_residue, 22: feature__protein__qual__function, 23: feature__protein__qual__g_o_component, 24: feature__protein__qual__name, 25: feature__

Resolved PCA/KMeans snapshot successfully.
Snapshot directory: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\snapshots\2026-04-11T22-17-07Z__q_891f443d754c
PCA coordinates path: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\snapshots\2026-04-11T22-17-07Z__q_891f443d754c\pca_coordinates_10D.npy
Cluster assignments path: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\snapshots\2026-04-11T22-17-07Z__q_891f443d754c\cluster_assignments.csv
Stability grid path: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\snapshots\2026-04-11T22-17-07Z__q_891f443d754c\stability_grid.csv


In [6]:
# =============================================================================
# CELL 6 — Print PCA/KMeans snapshot summary
# =============================================================================

pca_kmeans_manifest_file_sha256 = sha256_of_file(
    input_file_path=pca_kmeans_manifest_file_path,
)
cluster_assignments_file_sha256 = sha256_of_file(
    input_file_path=cluster_assignments_file_path,
)
stability_grid_file_sha256 = sha256_of_file(
    input_file_path=stability_grid_file_path,
)
selected_configuration_summary_dataframe = pd.DataFrame(
    [
        {
            "selected_pca_component_count": pca_kmeans_manifest_payload[
                "selected_pca_component_count"
            ],
            "selected_cluster_count_k": pca_kmeans_manifest_payload[
                "selected_cluster_count_k"
            ],
            "selected_variance_explained_fraction": pca_kmeans_manifest_payload[
                "selected_variance_explained_fraction"
            ],
            "selected_silhouette_best_sampled": pca_kmeans_manifest_payload[
                "selected_silhouette_best_sampled"
            ],
            "selected_init_ari_min": pca_kmeans_manifest_payload[
                "selected_init_ari_min"
            ],
            "selected_subsample_ari_min": pca_kmeans_manifest_payload[
                "selected_subsample_ari_min"
            ],
            "selection_reason": pca_kmeans_manifest_payload["selection_reason"],
            "final_sampled_silhouette_value": pca_kmeans_manifest_payload[
                "final_sampled_silhouette_value"
            ],
        }
    ]
)

print("PCA/KMeans snapshot is ready.")
print(
    f"Snapshot created at UTC: {pca_kmeans_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Sequence count: {pca_kmeans_manifest_payload['sequence_count']}")
print(f"PCA coordinates shape: {pca_coordinates.shape}")
print(f"Explained variance vector length: {explained_variance_ratio.shape[0]}")
print(f"Cluster assignments rows: {len(cluster_assignments_dataframe)}")
print(f"Stability grid rows: {len(stability_grid_dataframe)}")
print(f"Alignment report: {alignment_report}")
print(f"Cluster assignments SHA-256: {cluster_assignments_file_sha256}")
print(f"Stability grid SHA-256: {stability_grid_file_sha256}")
print(f"Manifest SHA-256: {pca_kmeans_manifest_file_sha256}")

display(selected_configuration_summary_dataframe)

PCA/KMeans snapshot is ready.
Snapshot created at UTC: 2026-04-11T22:17:07Z
Sequence count: 41345
PCA coordinates shape: (41345, 10)
Explained variance vector length: 10
Cluster assignments rows: 41345
Stability grid rows: 45
Alignment report: {'duplicate_protein_uid_in_metadata_count': 0, 'duplicate_protein_uid_in_sequence_count': 0, 'matched_metadata_row_count': 41345, 'metadata_row_count': 41345, 'missing_metadata_row_count': 0, 'missing_protein_uid_in_sequence_count': 0, 'sequence_row_count': 41345}
Cluster assignments SHA-256: 1979472611dda2f541ffb92d0cc909285638abfa97d4fabb2397376fb6bec303
Stability grid SHA-256: 6e730611f95063de2cf6b58a43d7b05ba8ba0d1d136a0840873bc13b7cfd1d3f
Manifest SHA-256: 950899147197e32cde90267502c6c36ab63bb2149b5daef2677ecc1d51949a8f


,selected_pca_component_count,selected_cluster_count_k,selected_variance_explained_fraction,selected_silhouette_best_sampled,selected_init_ari_min,selected_subsample_ari_min,selection_reason,final_sampled_silhouette_value
0,10,10,0.133266,0.296472,0.587121,0.471061,fallback_max_composite_score,0.296472


In [7]:
# =============================================================================
# CELL 7 — Preview ranked configurations from the stability grid
# =============================================================================

stability_grid_ranked_dataframe = stability_grid_dataframe.sort_values(
    [
        "composite_score_silhouette_times_min_ari",
        "silhouette_best_sampled_filled",
        "variance_explained_fraction",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

print("Top ranked PCA/KMeans configurations:")
display(stability_grid_ranked_dataframe.head(10))

Top ranked PCA/KMeans configurations:


,pca_component_count,variance_explained_fraction,k,silhouette_best_sampled,init_ari_mean,init_ari_min,init_ari_max,init_ari_pair_count,subsample_ari_mean,subsample_ari_min,subsample_ari_max,subsample_ari_pair_count,subsample_fraction,subsample_point_count,mean_pairwise_intersection_size,silhouette_best_sampled_filled,init_ari_min_clipped,subsample_ari_min_clipped,composite_score_silhouette_times_min_ari
0,10,0.133266,10,0.296472,0.727177,0.587121,0.988007,15,0.636884,0.471061,0.842805,15,0.8,33076,26450.333333,0.296472,0.587121,0.471061,0.081995
1,10,0.133266,3,0.473381,0.832031,0.496889,1.000000,15,0.753355,0.264342,0.999042,15,0.8,33076,26450.333333,0.473381,0.496889,0.264342,0.062178
2,20,0.184715,10,0.242065,0.638635,0.500366,0.924996,15,0.650639,0.502577,0.791072,15,0.8,33076,26450.333333,0.242065,0.500366,0.502577,0.060873
3,20,0.184715,9,0.285652,0.502372,0.256439,0.908894,15,0.650506,0.496654,0.792560,15,0.8,33076,26450.333333,0.285652,0.256439,0.496654,0.036381
4,10,0.133266,8,0.237619,0.545967,0.306501,0.998750,15,0.625643,0.482338,0.944219,15,0.8,33076,26450.333333,0.237619,0.306501,0.482338,0.035129
5,50,0.273295,5,0.356780,0.624753,0.252925,0.955206,15,0.606323,0.353342,0.942442,15,0.8,33076,26450.333333,0.356780,0.252925,0.353342,0.031885
6,10,0.133266,7,0.321681,0.620962,0.411435,0.940376,15,0.556026,0.230430,0.863153,15,0.8,33076,26450.333333,0.321681,0.411435,0.230430,0.030498
7,10,0.133266,9,0.234666,0.547227,0.253135,0.797272,15,0.629232,0.480093,0.947321,15,0.8,33076,26450.333333,0.234666,0.253135,0.480093,0.028518
8,50,0.273295,8,0.190577,0.641718,0.434506,0.833760,15,0.521734,0.283158,0.719403,15,0.8,33076,26450.333333,0.190577,0.434506,0.283158,0.023447
9,20,0.184715,8,0.274141,0.500275,0.249675,0.754204,15,0.539954,0.299234,0.803233,15,0.8,33076,26450.333333,0.274141,0.249675,0.299234,0.020481


In [8]:
# =============================================================================
# CELL 8 — Preview cluster assignments and PCA outputs
# =============================================================================

cluster_assignment_preview_row_limit = 10
cluster_assignment_preview_dataframe = cluster_assignments_dataframe.head(
    cluster_assignment_preview_row_limit
).copy()
cluster_size_summary_dataframe = (
    cluster_assignments_dataframe["cluster_label"]
    .value_counts()
    .rename_axis("cluster_label")
    .reset_index(name="row_count")
    .sort_values("cluster_label")
    .reset_index(drop=True)
)
pca_coordinate_preview_dataframe = pd.DataFrame(
    np.asarray(pca_coordinates[:5]),
    columns=[f"pc{i + 1}" for i in range(pca_coordinates.shape[1])],
)

print("Cluster assignments preview:")
display(cluster_assignment_preview_dataframe)
print("Cluster size summary:")
display(cluster_size_summary_dataframe)
print("Selected PCA coordinate preview:")
display(pca_coordinate_preview_dataframe.iloc[:, : min(10, pca_coordinate_preview_dataframe.shape[1])])

Cluster assignments preview:


,sequence_index,record_id,description,sequence_length,protein_uid,gbseq__accession_version,gbseq__comment,gbseq__create_date,gbseq__definition,gbseq__division,...,feature__source__qual__serovar,feature__source__qual__specimen_voucher,feature__source__qual__strain,feature__source__qual__sub_species,feature__source__qual__sub_strain,feature__source__qual__type_material,cluster_label,pc1,pc2,pc3
0,0,protein_uid=1000250755|accession=KXK13845.1|le...,protein_uid=1000250755|accession=KXK13845.1|le...,709,1000250755,KXK13845.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ15_CFX003003232 [C...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,5,-0.648324,0.347293,0.799823
1,1,protein_uid=1000266463|accession=KXK28958.1|le...,protein_uid=1000266463|accession=KXK28958.1|le...,1043,1000266463,KXK28958.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ01_02401 [Candidat...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,9,2.243139,0.800627,4.006205
2,2,protein_uid=1000285434|accession=KXK47085.1|le...,protein_uid=1000285434|accession=KXK47085.1|le...,365,1000285434,KXK47085.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ10_BCD003000691 [B...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,0,1.997014,-0.723664,0.610842
3,3,protein_uid=1000285973|accession=KXK47585.1|le...,protein_uid=1000285973|accession=KXK47585.1|le...,684,1000285973,KXK47585.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: Piwi domain-containing protein [Bacteroid...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,9,2.116695,-0.236212,1.624122
4,4,protein_uid=1000287044|accession=KXK48581.1|le...,protein_uid=1000287044|accession=KXK48581.1|le...,713,1000287044,KXK48581.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ13_03613 [Chlorofl...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,1,-1.701771,0.899851,0.130960
5,5,protein_uid=1000371080|accession=WP_061113836....,protein_uid=1000371080|accession=WP_061113836....,191,1000371080,WP_061113836.1,"REFSEQ: This record represents a single, non-r...",24-FEB-2016,"pPIWI_RE module domain-containing protein, par...",BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,0,3.170251,1.616076,-1.154772
6,6,protein_uid=1000882329|accession=WP_061139231....,protein_uid=1000882329|accession=WP_061139231....,560,1000882329,WP_061139231.1,"REFSEQ: This record represents a single, non-r...",25-FEB-2016,MULTISPECIES: RNaseH domain-containing protein...,BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,2,5.747970,6.130516,-1.738163
7,7,protein_uid=1000882353|accession=WP_061139255....,protein_uid=1000882353|accession=WP_061139255....,223,1000882353,WP_061139255.1,"REFSEQ: This record represents a single, non-r...",25-FEB-2016,MULTISPECIES: pPIWI_RE module domain-containin...,BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,0,3.264802,1.901162,-1.305100
8,8,protein_uid=1000934900|accession=WP_061181381....,protein_uid=1000934900|accession=WP_061181381....,346,1000934900,WP_061181381.1,"REFSEQ: This record represents a single, non-r...",25-FEB-2016,restriction endonuclease-related protein [Pseu...,BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0.580699,-0.664712,-0.583393
9,9,protein_uid=1001028003|accession=KXO00930.1|le...,protein_uid=1001028003|accession=KXO00930.1|le...,703,1001028003,KXO00930.1,Annotation was added by the NCBI Prokaryotic G...,25-FEB-2016,hypothetical protein LS48_00135 [Aequorivita a...,BCT,...,NaN,NaN,D-24,NaN,NaN,type strain of Vitellibacter aquimaris,9,2.376131,-0.559231,1.858831


Cluster size summary:


,cluster_label,row_count
0,0,18320
1,1,5237
2,2,602
3,3,964
4,4,336
5,5,10174
6,6,362
7,7,279
8,8,776
9,9,4295


Selected PCA coordinate preview:


,pc1,pc2,pc3,pc4,pc5,pc6,pc7,pc8,pc9,pc10
0,-0.648324,0.347293,0.799823,-0.062545,-0.196449,0.107159,-0.021799,-0.070401,-0.075205,-0.147423
1,2.243139,0.800627,4.006205,-0.441228,-0.635110,-0.976650,1.040125,0.463975,0.009997,0.087789
2,1.997014,-0.723664,0.610842,0.138571,-0.066984,0.161978,0.843851,-0.487037,0.253620,-0.099004
3,2.116695,-0.236212,1.624122,-0.143065,0.023480,-0.209802,-0.537214,-0.195576,-0.157753,0.363767
4,-1.701771,0.899851,0.130960,-1.081217,-0.146696,-0.252972,-0.354641,-0.105787,-0.108670,-0.373267


In [9]:
# =============================================================================
# CELL 9 — Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- metadata_snapshot_directory")
print("- metadata_csv_file_path")
print("- sweep_genes_snapshot_directory")
print("- sweep_genes_embeddings_file_path")
print("- pca_kmeans_snapshot_directory")
print("- pca_coordinates_file_path")
print("- explained_variance_ratio_file_path")
print("- cluster_assignments_file_path")
print("- stability_grid_file_path")
print("- profiling_log_file_path")
print("- alignment_report_file_path")
print("- pca_kmeans_manifest_payload")
print("- pca_coordinates")
print("- explained_variance_ratio")
print("- cluster_assignments_dataframe")
print("- stability_grid_dataframe")
print("- stability_grid_ranked_dataframe")
print("- profiling_log_dataframe")
print("- alignment_report")
print("- selected_configuration_summary_dataframe")
print("- cluster_size_summary_dataframe")

Variables exposed for downstream notebooks:
- metadata_snapshot_directory
- metadata_csv_file_path
- sweep_genes_snapshot_directory
- sweep_genes_embeddings_file_path
- pca_kmeans_snapshot_directory
- pca_coordinates_file_path
- explained_variance_ratio_file_path
- cluster_assignments_file_path
- stability_grid_file_path
- profiling_log_file_path
- alignment_report_file_path
- pca_kmeans_manifest_payload
- pca_coordinates
- explained_variance_ratio
- cluster_assignments_dataframe
- stability_grid_dataframe
- stability_grid_ranked_dataframe
- profiling_log_dataframe
- alignment_report
- selected_configuration_summary_dataframe
- cluster_size_summary_dataframe
